# Figure 1: Problem Setup And TriShift

- Figure / panels: `Fig1a`, `Fig1b`, `Fig1c`
- Inputs: `artifacts/results/*`, `artifacts/results/norman/metrics_nearest.csv`
- Outputs: `artifacts/paper_figures/main/Fig1_MethodOverview/*`
- Run order: run before all result notebooks; this notebook only prepares dataset/task summaries for Figure 1
- Dataset selector: edit `SELECTED_SINGLE_DATASET_KEYS` and `INCLUDE_NORMAN_GENERALIZATION` in the parameter cell below
- Role: Main text


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

sns.set_theme(style='ticks', font_scale=1.0)
plt.rcParams['figure.dpi'] = 220
plt.rcParams['savefig.dpi'] = 220

from scripts.trishift.analysis._result_adapter import DEFAULT_PAYLOAD_ROOTS


## Plan

- `Fig1a` 和 `Fig1b` 是方法/任务示意，当前 notebook 只负责为这两部分准备数据统计，不负责直接生成矢量示意图。
- `Fig1c` 负责汇总主线数据集与任务版图，明确五个单扰动 benchmark 加上 `Norman combo/subgroup` 的 stress-test 位置。
- 下方状态表只检查结果目录是否齐备；如果某些 baseline 还没跑完，对应数据集会显示 `pending`。


In [ ]:
ALL_MAINLINE_DATASETS = [
    {'dataset_key': 'dixit', 'dataset_label': 'Dixit', 'task_family': 'single-perturbation', 'main_role': 'core benchmark'},
    {'dataset_key': 'adamson', 'dataset_label': 'Adamson', 'task_family': 'single-perturbation', 'main_role': 'core benchmark'},
    {'dataset_key': 'norman', 'dataset_label': 'Norman (single)', 'task_family': 'single-perturbation', 'main_role': 'core benchmark'},
    {'dataset_key': 'replogle_k562_essential', 'dataset_label': 'Replogle K562', 'task_family': 'single-perturbation', 'main_role': 'core benchmark'},
    {'dataset_key': 'replogle_rpe1_essential', 'dataset_label': 'Replogle RPE1', 'task_family': 'single-perturbation', 'main_role': 'core benchmark'},
]
AVAILABLE_SINGLE_DATASET_KEYS = [spec['dataset_key'] for spec in ALL_MAINLINE_DATASETS]
SELECTED_SINGLE_DATASET_KEYS = ['adamson', 'norman']  # edit here
INCLUDE_NORMAN_GENERALIZATION = True  # edit here

invalid_dataset_keys = [key for key in SELECTED_SINGLE_DATASET_KEYS if key not in AVAILABLE_SINGLE_DATASET_KEYS]
if invalid_dataset_keys:
    raise ValueError(f'Unknown dataset keys: {invalid_dataset_keys}. Available: {AVAILABLE_SINGLE_DATASET_KEYS}')

MAINLINE_DATASETS = [spec for spec in ALL_MAINLINE_DATASETS if spec['dataset_key'] in SELECTED_SINGLE_DATASET_KEYS]
if not MAINLINE_DATASETS:
    raise ValueError('SELECTED_SINGLE_DATASET_KEYS produced an empty dataset list.')

NORMAN_GENERALIZATION = {
    'dataset_key': 'norman',
    'dataset_label': 'Norman combo/subgroup',
    'task_family': 'hard generalization',
    'main_role': 'Fig4 hard-generalization module',
}
OUT_ROOT = repo_root / 'artifacts' / 'paper_figures' / 'main' / 'Fig1_MethodOverview'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print('Selected single-perturbation datasets:', SELECTED_SINGLE_DATASET_KEYS)
print('Include Norman hard-generalization module:', INCLUDE_NORMAN_GENERALIZATION)


In [ ]:
def dataset_status_row(dataset_key: str, dataset_label: str, task_family: str, main_role: str) -> dict[str, object]:
    trishift_root = DEFAULT_PAYLOAD_ROOTS['trishift'] / dataset_key
    external_roots = {
        name: (root / dataset_key) for name, root in DEFAULT_PAYLOAD_ROOTS.items() if name != 'trishift'
    }
    trishift_ready = trishift_root.exists()
    external_ready = any(path.exists() for path in external_roots.values())
    status = 'ready' if trishift_ready and external_ready else 'pending'
    return {
        'dataset_key': dataset_key,
        'dataset_label': dataset_label,
        'task_family': task_family,
        'main_role': main_role,
        'trishift_ready': bool(trishift_ready),
        'external_baseline_ready': bool(external_ready),
        'status': status,
    }

scope_rows = [dataset_status_row(**spec) for spec in MAINLINE_DATASETS]
if INCLUDE_NORMAN_GENERALIZATION:
    scope_rows.append(dataset_status_row(**NORMAN_GENERALIZATION))
scope_df = pd.DataFrame(scope_rows)
scope_df.to_csv(OUT_ROOT / 'fig1_dataset_scope.csv', index=False, encoding='utf-8-sig')
display(scope_df)

plot_df = scope_df.copy()
plot_df['status_code'] = plot_df['status'].map({'ready': 1, 'pending': 0}).fillna(0)
plt.figure(figsize=(9, 4.5))
ax = sns.barplot(data=plot_df, x='dataset_label', y='status_code', hue='task_family', dodge=False)
ax.set_ylim(-0.05, 1.05)
ax.set_yticks([0, 1], ['pending', 'ready'])
ax.set_xlabel('')
ax.set_ylabel('Result availability')
ax.set_title('Mainline dataset scope for the paper')
ax.tick_params(axis='x', rotation=25)
ax.legend(title='')
plt.tight_layout()
plt.savefig(OUT_ROOT / 'fig1_dataset_scope.png', bbox_inches='tight')
plt.close()


In [ ]:
if INCLUDE_NORMAN_GENERALIZATION:
    try:
        norman_metrics = pd.read_csv(repo_root / 'artifacts' / 'results' / 'norman' / 'metrics_nearest.csv')
        subgroup_counts = (
            norman_metrics[['condition', 'subgroup']]
            .drop_duplicates()
            .groupby('subgroup', as_index=False)
            .size()
            .rename(columns={'size': 'n_conditions'})
        )
    except Exception:
        subgroup_counts = pd.DataFrame([
            {'subgroup': 'single', 'n_conditions': np.nan},
            {'subgroup': 'seen2', 'n_conditions': np.nan},
            {'subgroup': 'seen1', 'n_conditions': np.nan},
            {'subgroup': 'seen0', 'n_conditions': np.nan},
        ])
else:
    subgroup_counts = pd.DataFrame(columns=['subgroup', 'n_conditions'])

subgroup_counts.to_csv(OUT_ROOT / 'fig1_norman_subgroup_counts.csv', index=False, encoding='utf-8-sig')
display(subgroup_counts)

plt.figure(figsize=(6.2, 4.0))
ax = plt.gca()
if subgroup_counts.empty:
    ax.text(0.5, 0.5, 'Norman hard-generalization module disabled', ha='center', va='center')
    ax.axis('off')
else:
    ax = sns.barplot(data=subgroup_counts, x='subgroup', y='n_conditions', order=['single', 'seen2', 'seen1', 'seen0'], color='#4C78A8')
    ax.set_xlabel('')
    ax.set_ylabel('Number of conditions')
    ax.set_title('Norman subgroup split used in Fig4')
    for patch in ax.patches:
        height = patch.get_height()
        if pd.notna(height):
            ax.text(patch.get_x() + patch.get_width() / 2, height, f'{int(height)}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(OUT_ROOT / 'fig1_norman_subgroup_counts.png', bbox_inches='tight')
plt.close()

print(OUT_ROOT)
